In [ ]:
import boto3
from braket.aws import AwsSession

# Setup environment

In [ ]:
sess = AwsSession()
region = "eu-north-1" # region needs to align with Braket device location
role = sess.get_default_jobs_role()
default_bucket = sess.default_bucket()

prefix = "qml"

NOTE: If the step above fails with "RuntimeError: No default jobs roles found. Please ..." first create a default role in the AWS Console as described [here](https://docs.aws.amazon.com/braket/latest/developerguide/braket-jobs-first.html).

In [ ]:
sts_client = boto3.client("sts")
account_id = sts_client.get_caller_identity()["Account"]

In [ ]:
account_id

# Build Cointainer

The quantum transfer learning approach used in this notebook is a hybrid classical-quantum algorithm, i.e. it uses alternatingly classical and quantum compute resources. Quantum compute resources are allocated through a job queue on Amazon Braket. Running a hybrid classical-quantum algorithm from the notebook instance would submit a quantum task to the queue on every classical-quantum iteration, and hence will lead to a potentially high walltime caused by accumulated queueing times. For avoiding high walltime the hybrid algorithm can be submitted as a [Amazon Braket hybrid job](https://aws.amazon.com/blogs/aws/introducing-amazon-braket-hybrid-jobs-set-up-monitor-and-efficiently-run-hybrid-quantum-classical-workloads/), which will give associated quantum tasks high priority once the job has started. 

In a Amazon Braket job the training code is executed in a container environment. Amazon Braket offers [prebuilt docker containers](https://github.com/aws/amazon-braket-containers) among others for QML with PyTorch which is used in this notebook. However, the pre-buit PyTorch container does not have the module torchvision installed, and hence, a custom container is used here. Setting up a custom container is also described in more detail [here](https://github.com/aws/amazon-braket-examples/tree/main/examples/hybrid_jobs/3_Bring_your_own_container).

A Dockerfile defines the environment the training script will run in. Here a prebuilt Amazon Braket PyTorch image is used as a baseline and the Python module torchvision is added to it.

In [ ]:
%%writefile Dockerfile
FROM 292282985366.dkr.ecr.us-east-1.amazonaws.com/amazon-braket-pytorch-jobs:1.8.1-cpu-py37-ubuntu18.04

RUN python3 -m pip install --upgrade pip
RUN python3 -m pip install amazon-braket-sdk==1.35.5 --upgrade
RUN python3 -m pip install torchvision==0.10.1

To make the custom container available to Amazon Braket, an container image is created and stored in a repository of AWS Elastic Container Registry (ECR). The default Amazon Braket jobs have access to repositories starting with `amazon-braket`. For different names the poliy of the default role needs to be adjusted.

In [ ]:
# create ECR
ecr_repository_name = "amazon-braket-my-qtc"
image_uri_byoc=f"{account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repository_name}:latest"

In [ ]:
!aws ecr create-repository --repository-name {ecr_repository_name} --region {region}

In [ ]:
!docker login -u AWS -p $(aws ecr get-login-password --region us-east-1) 292282985366.dkr.ecr.us-east-1.amazonaws.com
!docker login -u AWS -p $(aws ecr get-login-password --region {region}) {account_id}.dkr.ecr.{region}.amazonaws.com

Let's build the custom Docker image defined by the Dockerfile and push it to the ECR repository. This makes the image ready to be used by Amazon Braket.

In [ ]:
!docker build -t dockerfile .

In [ ]:
!docker tag dockerfile:latest {account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repository_name}:latest

In [ ]:
!docker push {account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repository_name}:latest